# 03 - Baseline CNN Model: FC-Siam-Conc

## Objectives

The objective of this notebook is to implement a Fully Convolutional Siamese Network with Feature Concatenation (FC-Siam-Conc) for binary change detection using the LEVIR-CD dataset.

The specific objectives are:

- Understand the architecture of FC-Siam-Conc.
- Implement reusable convolutional blocks.
- Construct the shared Siamese encoder.
- Develop the decoder with skip connections.
- Concatenate feature maps from paired images.
- Generate a binary change map.
- Verify the model using a forward pass.
- Compute the number of trainable parameters.

### Code

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

Device: cuda


In [3]:
class DoubleConv(nn.Module):
    """
    Double Convolution Block

    Conv → BatchNorm → ReLU
    Conv → BatchNorm → ReLU
    """

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.double_conv = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(out_channels),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(out_channels),

            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

In [4]:
block = DoubleConv(3, 64)

x = torch.randn(1, 3, 256, 256)

y = block(x)

print(y.shape)

torch.Size([1, 64, 256, 256])


In [5]:
class EncoderBlock(nn.Module):
    """
    Encoder Block

    DoubleConv
        ↓
    MaxPool
    """

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = DoubleConv(in_channels, out_channels)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):

        features = self.conv(x)

        pooled = self.pool(features)

        return features, pooled

In [6]:
encoder = EncoderBlock(3, 64)

x = torch.randn(1, 3, 256, 256)

features, pooled = encoder(x)

print("Features :", features.shape)
print("Pooled   :", pooled.shape)

Features : torch.Size([1, 64, 256, 256])
Pooled   : torch.Size([1, 64, 128, 128])


In [7]:
class DecoderBlock(nn.Module):
    """
    Decoder Block

    TransposeConv
        ↓
    Concatenate Skip Features
        ↓
    DoubleConv
    """

    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()

        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2
        )

        self.conv = DoubleConv(
            out_channels + skip_channels,
            out_channels
        )

    def forward(self, x, skip):

        x = self.up(x)

        x = torch.cat([x, skip], dim=1)

        x = self.conv(x)

        return x

In [8]:
decoder = DecoderBlock(
    in_channels=128,
    skip_channels=64,
    out_channels=64
)

x = torch.randn(1,128,128,128)

skip = torch.randn(1,64,256,256)

output = decoder(x, skip)

print(output.shape)

torch.Size([1, 64, 256, 256])


In [9]:
class Encoder(nn.Module):
    """
    Shared Encoder used for both input images.
    """

    def __init__(self):
        super().__init__()

        self.enc1 = EncoderBlock(3, 64)
        self.enc2 = EncoderBlock(64, 128)
        self.enc3 = EncoderBlock(128, 256)
        self.enc4 = EncoderBlock(256, 512)

        self.bottleneck = DoubleConv(512, 1024)

    def forward(self, x):

        f1, p1 = self.enc1(x)

        f2, p2 = self.enc2(p1)

        f3, p3 = self.enc3(p2)

        f4, p4 = self.enc4(p3)

        bottleneck = self.bottleneck(p4)

        return bottleneck, [f1, f2, f3, f4]

In [10]:
encoder = Encoder()

x = torch.randn(1, 3, 256, 256)

bottleneck, skips = encoder(x)

print("Bottleneck:", bottleneck.shape)

for i, skip in enumerate(skips):
    print(f"Skip {i+1}:", skip.shape)

Bottleneck: torch.Size([1, 1024, 16, 16])
Skip 1: torch.Size([1, 64, 256, 256])
Skip 2: torch.Size([1, 128, 128, 128])
Skip 3: torch.Size([1, 256, 64, 64])
Skip 4: torch.Size([1, 512, 32, 32])


In [11]:
class FCSiamConc(nn.Module):

    def __init__(self):
        super().__init__()

        # Shared encoder
        self.encoder = Encoder()

        # Decoder
        self.dec4 = DecoderBlock(
            in_channels=2048,
            skip_channels=1024,
            out_channels=512
        )

        self.dec3 = DecoderBlock(
            in_channels=512,
            skip_channels=512,
            out_channels=256
        )

        self.dec2 = DecoderBlock(
            in_channels=256,
            skip_channels=256,
            out_channels=128
        )

        self.dec1 = DecoderBlock(
            in_channels=128,
            skip_channels=128,
            out_channels=64
        )

        self.final = nn.Conv2d(
            64,
            1,
            kernel_size=1
        )

    def forward(self, before, after):

        bottleneck1, skips1 = self.encoder(before)

        bottleneck2, skips2 = self.encoder(after)

        x = torch.cat([bottleneck1, bottleneck2], dim=1)

        skip4 = torch.cat([skips1[3], skips2[3]], dim=1)
        skip3 = torch.cat([skips1[2], skips2[2]], dim=1)
        skip2 = torch.cat([skips1[1], skips2[1]], dim=1)
        skip1 = torch.cat([skips1[0], skips2[0]], dim=1)

        x = self.dec4(x, skip4)

        x = self.dec3(x, skip3)

        x = self.dec2(x, skip2)

        x = self.dec1(x, skip1)

        x = self.final(x)

        return x

In [12]:
model = FCSiamConc().to(DEVICE)

before = torch.randn(1, 3, 256, 256).to(DEVICE)

after = torch.randn(1, 3, 256, 256).to(DEVICE)

output = model(before, after)

print(output.shape)

torch.Size([1, 1, 256, 256])


In [13]:
total_params = sum(p.numel() for p in model.parameters())

trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")

Total Parameters     : 36,268,225
Trainable Parameters : 36,268,225


In [14]:
print(model)

FCSiamConc(
  (encoder): Encoder(
    (enc1): EncoderBlock(
      (conv): DoubleConv(
        (double_conv): Sequential(
          (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
          (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (5): ReLU(inplace=True)
        )
      )
      (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (enc2): EncoderBlock(
      (conv): DoubleConv(
        (double_conv): Sequential(
          (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
          (3

# Observations

The FC-Siam-Conc architecture was successfully implemented using PyTorch as the baseline model for binary change detection.

## Key Observations

- The architecture consists of a shared Siamese encoder, ensuring that both pre-change and post-change images are processed using identical feature extraction weights.
- Double convolution blocks were used throughout the encoder and decoder to improve feature representation.
- Skip connections preserve fine spatial information during decoding, improving localization of changed regions.
- Feature maps from the paired images were concatenated at multiple decoder stages, enabling the network to learn meaningful representations of structural changes.
- A successful forward pass verified that the implemented architecture produces a single-channel binary change probability map with the same spatial dimensions as the input image.
- The modular implementation improves readability, maintainability, and future extensibility for research experiments.

## Conclusion

The FC-Siam-Conc model has been successfully implemented and validated through a forward pass. This architecture will serve as the baseline CNN model for subsequent training and performance comparison with transformer-based architectures such as ChangeFormer.